#<mark>4번: 가상 데이터셋을 생성한 뒤, GridSearch와 RandomSearch 기법으로 하이퍼파라미터 튜닝을 진행하세요.

In [ ]:
# ---------------------------------------------------------
# 런타임 유형을 변경하고, 잘 적용되었는지 확인용입니다.
# ---------------------------------------------------------
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

Sat Jun 13 06:11:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models

In [ ]:
# ----------------------- 🤖 제미나이 -----------------------
# 계속 값이 바뀌는게 싫어서 시드 고정할 수 있게 하는 걸 알려달라고 부탁.
#
# 💡 [추가] 실행할 때마다 데이터가 바뀌지 않도록 시드 고정
# ---------------------------------------------------------

import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # 멀티 GPU 사용할 경우를 대비
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = False # 확실한 고정을 위해 차단
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 문제 조건에 맞게, 가상 데이터셋 생성
# ---------------------------------------------------------
num_classes = 10
imputs_shape = (3, 224, 224)

X_train = np.random.random((1000, 3, 224, 224)).astype(np.float32)
y_train = np.random.randint(num_classes, size= (1000,))

X_test = np.random.random((200, 3, 224, 224)).astype(np.float32)
y_test = np.random.randint(num_classes, size=(200,))

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = [(inputs.to(device), labels.to(device)) for inputs, labels in train_loader]
test_loader = [(inputs.to(device), labels.to(device)) for inputs, labels in test_loader]

In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# ResNet50 모델 생성
# ---------------------------------------------------------
base_model = models.resnet50(weights=None)
base_model = nn.Sequential(*list(base_model.children()))[:-2]

class CustomResNet50(nn.Module):
  def __init__(self, num_classes):
    super(CustomResNet50, self).__init__()
    self.base_model = base_model
    self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
    self.fc1 = nn.Linear(2048, 256)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(256, num_classes)
    self.softmax = nn.Softmax(dim=1)

  def forward(self,x):
      x = self.base_model(x)
      x = self.global_avg_pool(x)
      x = torch.flatten(x,1)
      x = self.fc1(x)
      x = self.relu(x)
      x = self.fc2(x)
      return x

model = CustomResNet50(num_classes = num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# epoch = 10 번 학습 진행
# ---------------------------------------------------------
num_epochs = 10

for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0
  for i, l in train_loader:
    optimizer.zero_grad()
    o = model(i)
    loss = criterion(o,l)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
  print(f'epoch: {epoch+1}, Loss: {running_loss/len(train_loader)}')

epoch: 1, Loss: 2.44034880399704
epoch: 2, Loss: 2.3291062712669373
epoch: 3, Loss: 2.286037527024746
epoch: 4, Loss: 2.223937712609768
epoch: 5, Loss: 2.07757231220603
epoch: 6, Loss: 2.0239135548472404
epoch: 7, Loss: 1.9227850660681725
epoch: 8, Loss: 1.8069228120148182
epoch: 9, Loss: 1.6841649152338505
epoch: 10, Loss: 1.4068590700626373


In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 정확도 측정
# 결과 :
# 학습률(lr) 0.001
# batch_size 32
# epoch 10번 학습시켰을 때
# 정확도 accuray 는 다음과 같다
# ---------------------------------------------------------
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for i, l in test_loader:
    o = model(i)
    _, predicted = torch.max(o, 1)
    total += l.size(0)
    correct += (predicted == l).sum().item()

accuracy = correct/total
print(f'accuracy : {accuracy*100}%')

accuracy : 5.5%


In [ ]:
# ----------------------- 🤖 제미나이 -----------------------
# PyTorch용 Grid Search 완성 코드는 아래와 같다
# ---------------------------------------------------------

# ----------------------- 🗣️ 내 정리  -----------------------
# 알게된 사실:
# 1) batch_size가 바뀌니까 다시 dataloader부터 만들어야한다.
# 2) [핵심2]를 참고하면 모델을 새로 정의하고 있음
#     --> 즉, 새로 바뀐 학습률과 배치에 맞춰 모델을 새로 정의하기 때문에
#         위에서 10번 학습한 모델과는 별개가 되고
#         이 코드에서 진행한 총 4번의 모델은 다 각각 별개로, 7번씩 학습한 모델이다.
# ---------------------------------------------------------

import copy

# 1. 탐색할 하이퍼파라미터 그리드 설정
learning_rates = [0.001, 0.01]
batch_sizes = [32, 64]
epochs = 7

best_accuracy = 0.0
best_params = {}
best_model_state = None

# 2. Grid Search 중첩 반복문 시작
for lr in learning_rates: # 0.001과 0.01 일 때 ---------------------| 즉, 0.001 & 32 / 0.001 & 64 / 0.01 & 32 / 0.01 & 64 총 4번 반복한다
    for bs in batch_sizes: # 32와 64일 때  -------------------------|
        print(f"\n🚀 현재 조합 테스트 중 -> [Learning Rate: {lr} | Batch Size: {bs}]")

        # 🔥 [핵심1] PyTorch에서는 배치가 바뀔 때마다 DataLoader를 새로 만들어야 합니다.
        train_loader_js = DataLoader(train_dataset, batch_size=bs, shuffle=True)
        test_loader_js = DataLoader(test_dataset, batch_size=bs, shuffle=False)

        # GPU 전환 파이프라인 (기존에 작성하신 방식 적용)
        train_loader_js = [(inputs.to(device), labels.to(device)) for inputs, labels in train_loader_js]
        test_loader_js = [(inputs.to(device), labels.to(device)) for inputs, labels in test_loader_js]

        # 🔥 [핵심2] 새로운 하이퍼파라미터로 학습하기 위해 모델과 옵티마이저를 매번 초기화합니다.
        model = CustomResNet50(num_classes=num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # --- 3. 모델 학습 (Train Loop) ---
        for epoch in range(epochs):
            model.train()
            for i, l in train_loader_js:
                optimizer.zero_grad()
                o = model(i)
                loss = criterion(o, l)
                loss.backward()
                optimizer.step()

        # --- 4. 모델 평가 (Evaluation Loop) ---
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for i, l in test_loader_js:
                o = model(i)
                _, predicted = torch.max(o, 1)
                total += l.size(0)
                correct += (predicted == l).sum().item()

        accuracy = (correct / total) * 100
        print(f"➡️ 결과 - Test Accuracy: {accuracy:.2f}%")

        # --- 5. 최고 성능 기록 및 업데이트 ---
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_params = {'learning_rate': lr, 'batch_size': bs}
            best_model_state = copy.deepcopy(model.state_dict()) # 가장 좋은 모델 가중치 저장

print("\n=========================================")
print("🏆 Grid Search 종료! 최적의 하이퍼파라미터 결과")
print("=========================================")
print(f"최고 정확도 (Best Accuracy): {best_accuracy:.2f}%")
print(f"최적의 조합 (Best Parameters): {best_params}")


🚀 현재 조합 테스트 중 -> [Learning Rate: 0.001 | Batch Size: 32]
➡️ 결과 - Test Accuracy: 8.50%

🚀 현재 조합 테스트 중 -> [Learning Rate: 0.001 | Batch Size: 64]
➡️ 결과 - Test Accuracy: 10.50%

🚀 현재 조합 테스트 중 -> [Learning Rate: 0.01 | Batch Size: 32]
➡️ 결과 - Test Accuracy: 7.50%

🚀 현재 조합 테스트 중 -> [Learning Rate: 0.01 | Batch Size: 64]
➡️ 결과 - Test Accuracy: 6.00%

🏆 Grid Search 종료! 최적의 하이퍼파라미터 결과
최고 정확도 (Best Accuracy): 10.50%
최적의 조합 (Best Parameters): {'learning_rate': 0.001, 'batch_size': 64}


In [ ]:
# ----------------------- 🤖 제미나이 -----------------------
# 질문:
# (교재를 보니까 grid와 random의 차이를 잘 모르겠음)
# gridserach 와 randomsearch의 차이가 뭔지 설명해줘
#
#
# 대답:
# 1. Grid Search (격자 탐색)
# 말 그대로 바둑판의 격자(Grid)처럼 우리가 정해준 후보들의 모든 조합을 촘촘하게 전부 다 직접 실행해보는 방식.
# 동작 방식:
# 예를 들어 learning_rates = [0.001, 0.01], batch_sizes = [32, 64]라고 지정하면,
# 컴퓨터는 (0.001, 32), (0.001, 64), (0.01, 32), (0.01, 64) 이렇게 4가지 조합을
# 하나도 빠짐없이 꼼꼼하게 다 학습시키고 평가합니다.
# 장점:
# 내가 지정한 범위 안에서는 무조건 가장 좋은 조합을 찾아냅니다. 꼼꼼하고 직관적입니다.
# 단점:
# 하이퍼파라미터 종류가 많아지면 조합 수가 기하급수적으로 늘어납니다.
# 약 후보가 5개씩 4종류라면 5 X 5 X 5 X 5 = 625번이나 모델을 처음부터 다시 학습시켜야 하므로 시간이 엄청나게 오래 걸립니다.
#
# 2. Random Search (무작위 탐색)
# 모든 조합을 다 보는 것이 아니라, 우리가 정한 범위 내에서 무작위(Random)로 값을 뽑아서 지정한 횟수만큼만 테스트하는 방식.
# 동작 방식:
# "Lr는 0.001에서 0.1 사이, Batch Size는 16에서 128 사이에서 아무거나 랜덤하게 딱 4개만 조합해서 테스트해줘!"라고
# 컴퓨터에게 시키는 것입니다. 컴퓨터는 그 범위 안에서 무작위로 숫자를 뚝딱 뽑아서 설정된 횟수만큼만 실험을 진행합니다.
# 장점:
# 시간과 비용을 내가 원하는 만큼 제어할 수 있습니다. (전체 조합이 10,000개여도 "난 10번만 랜덤으로 돌려볼래"가 가능합니다.)
# 또한, 촘촘한 격자 사이에 숨어있는, 내가 미처 생각지 못한 의외의 좋은 수치를 우연히 찾아내기도 합니다.
# 단점:
# 운에 의존하는 경향이 있어서, 운이 나쁘면 정말 최적의 하이퍼파라미터 조합을 비껴가고 엉뚱한 조합들만 테스트하다 끝날 수도 있습니다.
# ---------------------------------------------------------


In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 제미나이가 알려준 gridSearch 방식을 바탕으로 randomSearch 작성해보기
# ---------------------------------------------------------

epochs = 7

best_accuracy = 0.0
best_params = {}
best_model_state = None

num_searches = 5

for search_idx in range(num_searches):
  learning_rates = np.random.uniform(0.001, 0.01) #--------------------------------|-- 바뀌는 부분은 여기(학습률과 배치 선언) 뿐
  batch_sizes = int(np.random.choice([16, 32, 64, 128]))   #-----------------------|
  print(f"\n🚀 현재 조합 테스트 중 -> [Learning Rate: {learning_rates} | Batch Size: {batch_sizes}]")

  # 🔥 [핵심1] PyTorch에서는 배치가 바뀔 때마다 DataLoader를 새로 만들어야 합니다.
  train_loader_js = DataLoader(train_dataset, batch_size=batch_sizes, shuffle=True)
  test_loader_js = DataLoader(test_dataset, batch_size=batch_sizes, shuffle=False)

  # GPU 전환 파이프라인 (기존에 작성하신 방식 적용)
  train_loader_js = [(inputs.to(device), labels.to(device)) for inputs, labels in train_loader_js]
  test_loader_js = [(inputs.to(device), labels.to(device)) for inputs, labels in test_loader_js]

  # 🔥 [핵심2] 새로운 하이퍼파라미터로 학습하기 위해 모델과 옵티마이저를 매번 초기화합니다.
  model = CustomResNet50(num_classes=num_classes).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=learning_rates)

  # --- 3. 모델 학습 (Train Loop) ---
  for epoch in range(epochs):
      model.train()
      for i, l in train_loader_js:
          optimizer.zero_grad()
          o = model(i)
          loss = criterion(o, l)
          loss.backward()
          optimizer.step()

  # --- 4. 모델 평가 (Evaluation Loop) ---
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
      for i, l in test_loader_js:
          o = model(i)
          _, predicted = torch.max(o, 1)
          total += l.size(0)
          correct += (predicted == l).sum().item()

  accuracy = (correct / total) * 100
  print(f"➡️ 결과 - Test Accuracy: {accuracy:.2f}%")

  # --- 5. 최고 성능 기록 및 업데이트 ---
  if accuracy > best_accuracy:
      best_accuracy = accuracy
      best_params = {'learning_rate': learning_rates, 'batch_size': batch_sizes}
      best_model_state = copy.deepcopy(model.state_dict()) # 가장 좋은 모델 가중치 저장

print("\n=========================================")
print("🏆 Random Search 종료! 최적의 하이퍼파라미터 결과")
print("=========================================")
print(f"최고 정확도 (Best Accuracy): {best_accuracy:.2f}%")
print(f"최적의 조합 (Best Parameters): {best_params}")


🚀 현재 조합 테스트 중 -> [Learning Rate: 0.006629785051764433 | Batch Size: 16]
➡️ 결과 - Test Accuracy: 6.50%

🚀 현재 조합 테스트 중 -> [Learning Rate: 0.007182436612465144 | Batch Size: 64]
➡️ 결과 - Test Accuracy: 6.50%

🚀 현재 조합 테스트 중 -> [Learning Rate: 0.007181929783536824 | Batch Size: 16]
➡️ 결과 - Test Accuracy: 6.50%

🚀 현재 조합 테스트 중 -> [Learning Rate: 0.0065935984960020615 | Batch Size: 64]
➡️ 결과 - Test Accuracy: 8.50%

🚀 현재 조합 테스트 중 -> [Learning Rate: 0.005613877470675148 | Batch Size: 64]
➡️ 결과 - Test Accuracy: 9.00%

🏆 Random Search 종료! 최적의 하이퍼파라미터 결과
최고 정확도 (Best Accuracy): 9.00%
최적의 조합 (Best Parameters): {'learning_rate': 0.005613877470675148, 'batch_size': 64}


# <font color='red'>1. 배운 점</font>


<font color='green'>

1.   GridSearch의 핵심은 "매회 백지에서 다시 시작".
- GridSearch 반복문 안에서 `model = CustomResNet50(...)`을 매번 새로 선언하므로, 위에서 10번 학습한 모델과는 완전히 별개이며, 4개 조합이 각각 독립적으로 처음부터 학습됨.
- 만약 앞 조합의 학습 상태를 이어받으면 뒤 조합이 부당하게 덕을 보므로, 각 조합의 성능을 오롯이 그 조합만의 것으로 신뢰하려면 모델·옵티마이저를 매회 초기화해야 함을 확인. (일반적으로 같은 모델에 train을 다시 돌리면 가중치가 누적되어 총 20번 학습이 되지만, 여기서는 의도적으로 초기화한 것.)
2.   GridSearch vs RandomSearch의 차이
- GridSearch는 지정한 후보들의 모든 조합을 빠짐없이 격자처럼 탐색해 범위 내 최적을 보장하지만, 후보가 늘면 조합 수가 기하급수적으로 폭발(5종류×5개 = 625번)함. - RandomSearch는 범위만 정하고 지정한 횟수만큼만 무작위 추출해, 시간을 직접 제어할 수 있고 격자 사이에 숨은 의외의 좋은 값을 찾기도 하지만 운에 의존함.


# <font color='red'>2. 어려웠던 점</font>

<font color='green'>1.   ResNet50의 출력 채널 차원 불일치
- fc1 입력을 512로 적었다가 RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x2048 and 512x256)를 마주.
- ResNet50은 Bottleneck 블록이 마지막에 채널을 4배 확장해 layer4 출력이 2048(512×4)이 되므로, GAP·flatten을 거쳐 들어오는 벡터가 2048인데 fc1이 512를 기다려 행렬 곱이 불가능했던 것.
- fc1을 `nn.Linear(2048, 256)`으로 수정해 해결.
2.   RandomSearch를 정답 없이 직접 구현
- GridSearch 코드를 토대로, 중첩 for문을 탐색 횟수 단일 루프(`for search_idx in range(num_searches)`)로 바꾸고, 루프 첫 줄에서 `np.random.uniform()`(학습률)과 `np.random.choice()`(배치)로 조합을 매회 무작위 추출하도록 직접 작성. 정답 코드 없이 힌트만으로 구조를 옮기는 과정에서 GridSearch와 RandomSearch가 "후보 나열 vs 범위 추출"이라는 한 지점에서만 갈린다는 것을 체득.

# <font color='red'>3. 앞으로의 보완점</font>


<font color='green'> 1.   random 추출 조정

- randomSearch에서 정말 random으로 뽑으니 해당 실행에서는 128이라는 선택지를 줬음에도 실행되지 않음을 확인. 또한, 학습률 또한 점차 줄어드는 방향이 아니라 무조건 randomd으로 뽑히고 있음.
- randomSearch에서 다른 학습률과 배치임에도 불구하고3번이나 accuracy가 동일하게 나옴. 확인 결과, 가상데이터셋을 사용했으며 + 하이퍼파라미터(학습률)이 너무 커서 그런 것이라고
- 앞으로 random 중에서도 어떤 식으로 random하게 뽑게 할건지 조건을 걸어보면 좋을 것 같다.


# <font color='red'>4. 수행 과정 종합 평가</font>


<font color='green'>1.   서론
- 본 과제는 무작위로 생성한 가상 데이터셋(1000장 학습 / 200장 테스트, 10클래스, 224×224)을 ResNet50으로 학습시키는 환경에서, GridSearch와 RandomSearch 두 가지 하이퍼파라미터 튜닝 기법을 직접 구현해 비교하는 것을 목표로 진행되었습니다. Google Colab GPU 환경에서 수행했습니다.

<font color='green'>2.   본론


작업은 환경 고정 → 기준 학습 → 두 탐색 기법 구현 순으로 전개되었습니다.

- 환경 고정 및 기준 학습:
실행마다 데이터가 바뀌는 문제를 `set_seed(42)`로 네 종류의 난수 생성기를 모두 묶어 해결한 뒤, lr=0.001·batch=32·10에포크 조건으로 ResNet50을 1회 기준 학습했습니다.
- 이 과정에서 fc1의 입력 차원을 2048로 맞추는 디버깅을 거쳤습니다.
- 가상 데이터가 완전 무작위라 정확도가 10% 내외에 머무는 것임을 확인했으며, 본 과제의 목표는 정확도 수치 자체가 아니라 탐색 코드가 에러 없이 작동하고 best_params가 정상 산출되는지에 있음을 인지했습니다.
- GridSearch 구현:
  *   학습률 [0.001, 0.01] × 배치 [32, 64]의 4개 조합 전체를 중첩 반복문으로 탐색했습니다. 각 조합마다 DataLoader를 새로 만들고 모델·옵티마이저를 초기화한 뒤 7에포크씩 학습해, 최고 성능 가중치와 조합을 기록하도록 구성했습니다.
- RandomSearch 구현:
  *   GridSearch 구조를 토대로, 정답 코드 없이 힌트만 받아 직접 작성했습니다. 중첩 루프를 탐색 횟수 단일 루프로 바꾸고, 매 루프 첫 줄에서 학습률은 연속 범위(0.001~0.01)에서, 배치는 후보 리스트에서 무작위 추출하도록 변경했습니다.

<font color='green'>3.   결론
- 동일한 ResNet50 모델에 대해 두 가지 탐색 기법을 직접 구현하며, GridSearch는 "모든 격자를 빠짐없이", RandomSearch는 "범위 안을 지정 횟수만큼 무작위로" 탐색한다는 차이를 코드 수준에서 체득했습니다.
- 특히 두 기법이 사실상 '하이퍼파라미터를 후보로 나열하느냐, 범위에서 뽑느냐'와 '중첩 루프냐 단일 루프냐'라는 두 지점에서만 갈린다는 것을 직접 변환하며 확인했습니다.
- 가장 값진 수확은 "탐색의 공정성"에 대한 이해였습니다. 각 조합을 `백지 상태에서 동일한 에포크로 학습`시켜야만 그 조합의 성능을 오롯이 신뢰할 수 있다는 원리는, 앞으로 모든 모델 비교 실험에 적용할 기준이 되었습니다.



🤖 제미나이 대화 원본 : https://gemini.google.com/share/2ea86c22fffd


수행시간 : **2시간 10분소요**